# N054 · 单调栈与最近更大更小元素

**目标：** 依据支配关系移除不可能再有用的候选。

**先修：** N014, N051。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 索引栈；严格/非严格；左右边界；循环数组；均摊。

**配套讲解来源：** [同名逐章意见](../../comment/054_monotonic_stack_neighbors.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这类问题的统一形态是：**对每个位置，找它右边（或左边）第一个比它大/小的元素**。两个代表：

1. **每日温度**：每天给你一个温度，问每个日子要再等几天才遇到更暖的一天。输入 `[73,74,75,71,69,72,76,73]` → 输出 `[1,1,4,2,1,1,0,0]`。为什么：第 0 天 73 度，第二天 74 度更暖，等 1 天；第 2 天 75 度，后面一直更冷，直到第 6 天 76 度，等 4 天；最后两天的右边再没有更暖的，填 0。
2. **循环数组的下一个更大元素**：数组首尾相接当成环，每个数往右绕着找第一个严格更大的数，找不到填 -1。输入 `[1,2,1]` → 输出 `[2,-1,2]`。为什么：第 0 个数 1 的右边是 2，答案 2；第 1 个数 2 绕一圈也找不到比自己大的，答案 -1；第 2 个数 1 绕过尾部回到开头的 1 不算（要**严格更大**），再往前是 2，答案 2。

先看暴力为什么慢：对每个下标 i，向右一个一个试 j=i+1, i+2, ...，找到第一个 `t[j]>t[i]` 就停。数组整体递减时（比如 `[75,74,73,72,71]`）每个位置都要试到最右端，总比较次数约 n²/2；n=10⁵ 时约 50 亿次比较，远超时限。单调栈的改进思路是：**让每个下标只在"答案出现的那一刻"被处理一次**——没等到答案之前它安静地在栈里排队，答案一来立刻结算离场，一生只进栈一次、出栈一次，所以总共 O(n)。

## 2. 基础知识：先读懂这些词

- **单调栈（monotonic stack）**：一个普通的栈，但我们随时维持它从底到顶具有某种单调性（本章是"对应的温度非递增"）。它不是新数据结构，而是栈的一种使用纪律。
- **索引栈**：栈里存的是**下标**而不是温度值。因为"等几天"（`i-j`）和"回填到答案数组的哪个格子"都需要下标，温度值随时可以用 `t[stack[-1]]` 查到。
- **支配关系（dominance）**：如果 j 在 i 左边且 `t[j] <= t[i]`（i 更暖或一样暖），那 j 对"找更暖日子"这件事就永远没用了——任何能被 j 满足的查询，i 都能满足而且更近。所以 i 一来，j 就可以永久弹掉。“学习目标”部分 的目标"依据支配关系移除不可能再有用的候选"说的就是这个。
- **严格 / 非严格**：`t[stack[-1]] < x` 是严格比较——相等不弹、相等不算"更暖"。如果把 `<` 写成 `<=`，`[5,5,5]` 的答案会从 `[0,0,0]` 变成 `[1,1,0]`，题意就错了。“正确性与复杂度”部分 特意警告："严格更大不能写成大于等于"。
- **左右边界**：同一套栈换个扫描方向就能找"左边第一个更小/更大"。往右扫解决"下一个"，往左扫（或从右往左扫）解决"上一个"——练习 1 会让你自己推一遍。
- **循环数组（环形数组）**：把数组首尾拼成环，越过末尾就回到开头继续找。技巧是"逻辑上扫两圈"：下标 `i` 走 `0..2n-1`，真实下标是 `i%n`。
- **均摊（amortized）**：虽然某个元素可能被一次 while 弹很多个，但每个下标一生只入栈一次、出栈至多一次，把总弹出次数加起来是 n，平均每次循环摊下来是 O(1)。

## 3. 思路推导：从小例子开始

**daily_temperatures 的步骤：**

1. 开一个答案数组全部填 0（0 就是"永远等不到"的默认值，所以没被弹出的索引不用管）。
2. 从左到右扫，第 i 天到来时，看栈顶：栈顶那天 j 的温度比今天低，说明**今天就是 j 等了半天的更暖日**，弹出 j，写入 `answer[j]=i-j`（隔了几天）。
3. 一直弹到栈顶温度不低于今天为止（它们还得继续等更暖的），然后把今天的下标 i 压栈，排队等它的更暖日。

**手算演示（“运行示例”部分 的表格，t=[73,74,75,71,69,72,76,73]）：**

| i | 温度 | 本次解决的索引 | 尚待解决栈（下标） |
|---|------|----------------|---------------------|
| 0 | 73 | （无） | [0] |
| 1 | 74 | 0 | [1] |
| 2 | 75 | 1 | [2] |
| 3 | 71 | （无） | [2, 3] |
| 4 | 69 | （无） | [2, 3, 4] |
| 5 | 72 | 4, 3 | [2, 5] |
| 6 | 76 | 5, 2 | [6] |
| 7 | 73 | （无） | [6, 7] |

逐行看几个关键点：第 5 天 72 度一次解决了 69 和 71（它们都比 72 小，都被弹掉），但 75 不比 72 小，留在栈里继续等；第 6 天 76 度把 72 和 75 一起解决。

把表里的弹栈事件逐条翻译成答案数组：下标 0 在 i=1 被弹 → `answer[0]=1-0=1`；下标 1 在 i=2 被弹 → `answer[1]=1`；下标 4、3 在 i=5 被弹 → `answer[4]=5-4=1`、`answer[3]=5-3=2`；下标 5、2 在 i=6 被弹 → `answer[5]=1`、`answer[2]=6-2=4`；下标 6、7 从未被弹 → 保持默认 0。拼起来正好 `[1,1,4,2,1,1,0,0]`，和测试断言完全对上。

再看一个"越等越久"的极端形状：`t=[75,74,73,72,71,80]`。前五天温度一路走低，谁也解决不了谁，栈越堆越高（下标 0 到 4 全在栈里）；第 6 天 80 度一来，while 循环连续弹 5 次，一天之内把所有人的答案全部结清（4,3,2,1,1），最后栈里只剩自己。这个例子说明：结算可以极度不均匀——前面五步一次都不弹，最后一步弹五次，但每个下标终究只被弹一次，总账仍是 O(n)。

**next_greater_circular 的步骤：**

1. 答案全填 -1（找不到的默认值）。
2. 下标 i 从 0 走到 2n-1，真实数字是 `nums[i%n]`——第二圈就是在"绕回开头继续找"。
3. 同样的弹栈规则：新数严格更大就弹，弹出者直接得到答案 `x`（这题要的是值不是天数）。
4. 关键一行 `if i<n: stack.append(i)`——**只有第一圈才入栈**。第二圈只负责"用开头的数去解决尾部剩下的人"，如果第二圈也入栈，同一个下标会进栈两次，答案可能被写坏。

**手算演示：`[1,2,1]`（n=3，共扫 6 步）——**

| i | x=nums[i%3] | 弹出（原因） | 入栈？ | 栈 | answer |
|---|-------------|----------------|--------|-----|--------|
| 0 | 1 | —（栈空） | 是（i<3） | [0] | [-1,-1,-1] |
| 1 | 2 | 0（1<2，answer[0]=2） | 是 | [1] | [2,-1,-1] |
| 2 | 1 | —（2 不大于 1） | 是 | [1,2] | [2,-1,-1] |
| 3 | 1 | —（栈顶 1 不小于 1，严格） | 否（i≥3） | [1,2] | 不变 |
| 4 | 2 | 2（1<2，answer[2]=2） | 否 | [1] | [2,-1,2] |
| 5 | 1 | —（2 不小于 1） | 否 | [1] | 不变 |

三个下标的结局各不相同：下标 0 在第一圈就被解决；下标 2 靠第二圈（i=4 绕回到值 2）才解决；下标 1 是全环最大值，两圈扫完仍在栈里，保持默认 -1。第二圈的价值就是把"答案绕过数组尾部"的情况补齐，而"只出不进"保证没人被重复处理。

## 4. 核心代码：daily_temperatures

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def daily_temperatures(t):
    stack = []
    answer = [0] * len(t)
    for i, x in enumerate(t):
        while stack and t[stack[-1]] < x:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer
```

### 逐段读懂代码

**每日温度（对应实现）：**

```python
def daily_temperatures(t):
    stack = []
    answer = [0] * len(t)
    for i, x in enumerate(t):
        while stack and t[stack[-1]] < x:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer
```

- `answer=[0]*len(t)`：默认 0 意味着"到扫描结束还没找到更暖日的日子不用再处理"，它们保持 0。
- `while stack and t[stack[-1]]<x` 是核心：条件有两半——栈非空、栈顶那天的温度**严格小于**今天。注意查值用的是 `t[stack[-1]]`（栈里存下标，要回到数组取值），这就是"索引栈"的代价与好处。
- `answer[j]=i-j`：j 是等待者的下标，i 是更暖日的下标，差就是"等了几天"。
- 一个 while 弹一串：今天更暖可能同时是好几天的答案（比如 76 解决了 72 和 75），所以用 while 不用 if。
- 循环末尾 `stack.append(i)` 无条件压栈：今天自己也要开始等待。
- 为什么栈保持"温度非递增"？因为比今天小的都被弹光了，今天压进去时栈里剩下的都不小于它——这是算法的自然结果，不需要额外维护。

**循环数组版（对应实现）：**

```python
def next_greater_circular(nums):
    n = len(nums)
    answer = [-1] * n
    stack = []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        if i < n:
            stack.append(i)
    return answer
```

- `for i in range(2*n)` + `nums[i%n]`：逻辑扫描两圈，第二圈的下标 3、4、5 经过 `%n` 映射回 0、1、2——这就是"跨过尾部绕回开头"的全部实现。
- 弹栈那行直接写答案 `answer[stack.pop()]=x`：这题要"下一个更大元素的值"，被更大的 x 弹出时 x 就是答案，不需要减下标。
- `if i<n: stack.append(i)` 是最容易写错的一行：第二圈**只出不进**。假如第二圈也入栈，下标 0 可能第二次进栈，而它早已有答案，逻辑全乱。“思路推导”部分 特意强调"只在第一圈入栈，避免同一索引重复入栈"。
- 默认值 -1（不是 0）：找不到更大的填 -1，与"值恰好是 0"区分开。
- 为什么第二圈不需要把弹栈条件也改一遍？因为 `nums[stack[-1]]<x` 本身对"绕回来的数"一视同仁——第二圈的 x 就是数组开头的数，比较逻辑完全一样，唯一的差别只是不再产生新的等待者。

另外注意两个函数的分工差异：`daily_temperatures` 弹栈时写的是**天数差** `i-j`（题目问等几天），`next_greater_circular` 弹栈时写的是**元素值** `x`（题目问下一个更大数是谁）。栈的运转逻辑一模一样，只有"结算时写什么进答案"不同——这也是单调栈题的通用套路：框架不变，改的是结算表达式。

## 5. 分段实现：按顺序运行

**本章接口：** `daily_temperatures(t)`、`next_greater_circular(nums)`

### daily_temperatures

In [1]:
def daily_temperatures(t):
    stack = []
    answer = [0] * len(t)
    for i, x in enumerate(t):
        while stack and t[stack[-1]] < x:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer

### next_greater_circular

In [2]:
def next_greater_circular(nums):
    n = len(nums)
    answer = [-1] * n
    stack = []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        if i < n:
            stack.append(i)
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

t=[73,74,75,71,69,72,76,73]; stack=[]; rows=[]
for i,x in enumerate(t):
    popped=[]
    while stack and t[stack[-1]]<x: popped.append(stack.pop())
    stack.append(i); rows.append((i,x,popped,stack[:]))
show_table(['i','温度','本次解决的索引','尚待解决栈'],rows)

i,温度,本次解决的索引,尚待解决栈
0,73,[],[0]
1,74,[0],[1]
2,75,[1],[2]
3,71,[],"[2, 3]"
4,69,[],"[2, 3, 4]"
5,72,"[4, 3]","[2, 5]"
6,76,"[5, 2]",[6]
7,73,[],"[6, 7]"


## 7. 正确性、适用条件与复杂度

栈值非递增，栈内索引递增。弹出的每个元素此前没有更大值，现在首次遇到；每索引仅入栈一次、出栈至多一次。

**代价：** O(n)时间、O(n)空间；严格更大不能写成大于等于。

### 展开理解

- **为什么每个被弹出者的答案是对的**：“正确性与复杂度”部分 说"弹出的每个元素此前没有更大值，现在首次遇到"。展开讲：某个下标 j 能一直躺在栈里，说明从 j+1 到 i-1 之间没有任何一天温度超过 t[j]——真有的话 j 早就被弹走了。所以 i 是**第一个**比 t[j] 大的，`i-j` 恰好就是等待天数。反过来，新来的 x 一旦大于栈顶，它对栈顶而言就是"等到了"，弹掉没有任何遗憾。
- **为什么没被弹出的填 0/-1 是对的**：它们右边没有任何更大的元素（循环版则绕了一整圈都没有），默认值正是"找不到"的含义。
- **为什么是严格比较**：`t[stack[-1]]<x` 保证相等温度不算"更暖"。如果改成 `<=`，`[5,5,5]` 里第 0 天会被第 1 天弹掉（5 计为比 5 大），答案变 `[1,1,0]`，与题意（要更暖）不符。测试里专门放了 `[5,5,5]` 这条守门。
- **复杂度**：每个下标最多进栈一次、出栈一次，while 循环的总弹出次数加起来不超过 n，所以整个扫描是 O(n) 时间、O(n) 空间（栈和答案数组）。具体感受一下：n=10⁵ 个温度，双指针暴力法最坏约 10¹⁰/2 次比较（要算几十秒），单调栈只有约 2×10⁵ 次栈操作（几毫秒）。循环版多扫的一圈不改变量级：总步数是 2n，仍是 O(n)。
- **失效条件（“掌握检查”部分 让你想的那类问题）**：本章实现假设"更暖"是严格大于且温度可比。如果题目改成"找第一个不低于今天的温度"（大于等于），把弹栈条件换成 `<=` 即可；如果数组里有 None 或字符串这种不可比较的值，`t[stack[-1]]<x` 会直接抛 TypeError——先修（N008）讲过，比较运算只在同类型之间成立。

顺手列一份本章常见踩坑清单，写代码前对照一遍：

- 栈里存了值而不是下标：算不出 `i-j`，也回填不了答案数组。
- 弹栈条件把 `<` 写成 `<=`：`[5,5,5]` 直接翻车（第六节的守门断言就是为它准备的）。
- 循环版第二圈忘记 `if i<n`：下标重复入栈，答案被二次覆盖。
- 忘了给 answer 预填默认值（0 或 -1），指望"没弹到的后面再补"：循环结束后根本没有"后面"。
- while 写成 if：一次只能弹一个，`[73,74,75,71,69,72,76,73]` 里 76 要连弹两个的场景就错了。
- 空数组没有特判：`range(2*0)` 循环体一次都不执行，返回空列表，倒是天然安全——但你要能说清为什么安全。

## 8. 单元测试：每个用例在检查什么

- `assert daily_temperatures([73,74,75,71,69,72,76,73])==[1,1,4,2,1,1,0,0]`：正常情况，含"连续下降再猛升"（一次弹多个）和"尾部无解填 0"两种形态。
- `assert daily_temperatures([5,5,5])==[0,0,0]`：特殊值——全部相等。专测严格性：相等不算更暖。这条是把 `<` 误写成 `<=` 时的照妖镜。
- `assert next_greater_circular([1,2,1])==[2,-1,2]`：循环数组的三个面——绕回开头找到（第 2 个数）、全环最大无解填 -1（第 1 个数）、严格更大（末尾的 1 不能被开头的 1 满足，要再绕到 2）。
- 最后的双重 for（`for n in range(6): for a in product([0,1,2],repeat=n)`）：**穷举测试**。对长度 0~5 的所有 3^n 个小数组（共 1+3+9+27+81+243=364 个），用一行暴力参照（`next((...) for d in range(1,n) if ...)` 从近到远找第一个更大）逐个对答案。小数组上穷举能撞出所有"相等、回绕、最大值、空数组"的组合，比任何单条手写用例都可靠。读懂那行参照代码也有收获：`next((a[(i+d)%n] for d in range(1,n) if a[(i+d)%n]>a[i]),-1)` 的意思是"按 d=1,2,...,n-1 的距离从近到远试，取第一个满足大于的位置上的值，一个都没有就取 -1"——这行暴力本身就是题意的直译，拿它对照单调栈版本，最能看清"优化版没有改变答案、只是改变了找答案的路线"。

顺带一提空数组的情况：n=0 时 product 只产生一个空元组，参照式退化为 -1 列表，`next_greater_circular([])` 返回 `[]`（准确说是空列表，因为 `answer=[-1]*0`）……实际上 n=0 时 range(2*n) 也是空的，直接返回空列表，两侧一致。穷举把这种"写代码时根本想不到的输入"也自动测了。

In [4]:
assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]

assert daily_temperatures([5, 5, 5]) == [0, 0, 0]

assert next_greater_circular([1, 2, 1]) == [2, -1, 2]

from itertools import product

for n in range(6):
    for a in product([0, 1, 2], repeat=n):
        ref = []
        for i in range(n):
            ref.append(next((a[(i + d) % n] for d in range(1, n) if a[(i + d) % n] > a[i]), -1))
        assert next_greater_circular(a) == ref

print('N054: 所有本章断言通过')

N054: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导前一个更小元素。

**练习 2：** 解释相等元素如何处理。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（前一个更小元素）**：提示：把问题镜像——你要找的是"左边第一个比它小的"。两条路线任选：(a) 从右往左扫，栈里存"还没等到右边查询者的候选"，比较方向翻转；(b) 仍从左到右，新元素 x 到来时，栈顶大于等于 x 的都可以弹掉（它们被 x 支配），弹剩下的栈顶就是 x 的前一个更小元素——注意这种写法答案在**入栈前**读取，和本章"入栈后等未来弹出"正好相反。手算用 `[3,1,4,2]`：答案应是 `[-1,-1,1,1]`。
- **练习 2（相等元素怎么处理）**：提示：先用 `[5,5,5]` 做实验，把 `t[stack[-1]]<x` 改成 `<=`，观察 `daily_temperatures` 的答案怎么变（`[0,0,0]` → `[1,1,0]`）。然后总结规则：弹不弹相等的元素，由题意决定——"严格更暖"用 `<`，"大于等于即可"用 `<=`；同时它还决定栈的单调性是"非递增"还是"严格递减"，以及相等的一串元素会不会被同一个后来者一次性弹光。边界例子建议用 `[2,2,3]`。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def daily_temperatures(t):
    stack = []
    answer = [0] * len(t)
    for i, x in enumerate(t):
        while stack and t[stack[-1]] < x:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer

def next_greater_circular(nums):
    n = len(nums)
    answer = [-1] * n
    stack = []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        if i < n:
            stack.append(i)
    return answer

# 示例与回归测试
assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]

assert daily_temperatures([5, 5, 5]) == [0, 0, 0]

assert next_greater_circular([1, 2, 1]) == [2, -1, 2]

from itertools import product

for n in range(6):
    for a in product([0, 1, 2], repeat=n):
        ref = []
        for i in range(n):
            ref.append(next((a[(i + d) % n] for d in range(1, n) if a[(i + d) % n] > a[i]), -1))
        assert next_greater_circular(a) == ref

print('N054: 所有本章断言通过')

N054: 所有本章断言通过


## 11. 代表题与迁移

- **739. Daily Temperatures**（[官方入口](https://leetcode.com/problems/daily-temperatures/)）：练"索引栈 + 答案存天数差"的最标准单调栈，`daily_temperatures` 就是本题原型。
- **496. Next Greater Element I**：练"下一个更大元素的值"与哈希表缩小查询范围的变体，是 `next_greater_circular` 的非循环简化版。
- **503. Next Greater Element II**：练循环数组的"扫两圈、只一圈入栈"技巧，`next_greater_circular` 直接对应本题。

三道题共用同一个栈框架，差别只在三处：找"下一个"还是"上一个"、要值还是要下标差、数组是不是环形。把这三处开关在心里列成表格，单调栈这一章就算真的学通了。

下一章（N055）会把"找到邻居"升级成"算出邻居围出的范围有多大"，栈的用法从查询变成结算，值得带着这个预期去读。

**题面入口：** [739](https://leetcode.com/problems/daily-temperatures/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。